In [5]:
import pandas as pd
import numpy as np

# Load new file
df = pd.read_csv("cbb_data.csv")

# Clean headers
df.columns = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()

# Ensure numeric
df["CLOSING SPREAD"] = pd.to_numeric(df["CLOSING SPREAD"], errors="coerce")
df["CLOSING TOTAL"]  = pd.to_numeric(df["CLOSING TOTAL"], errors="coerce")
df["Team W/L (1/0)"] = pd.to_numeric(df["Team W/L (1/0)"], errors="coerce")
df["Team ML + Opp Team Over Win/Loss (1/0)"] = pd.to_numeric(
    df["Team ML + Opp Team Over Win/Loss (1/0)"], errors="coerce"
)

# Favorite rows only (1 row per game)
fav = df[df["CLOSING SPREAD"] < 0].copy()

games = (
    fav.groupby("GAME-ID", as_index=False)
       .agg(
           TOTAL=("CLOSING TOTAL", "first"),
           FAV_WIN=("Team W/L (1/0)", "first"),
           FAV_WIN_OPP_OVER=("Team ML + Opp Team Over Win/Loss (1/0)", "first")
       )
)

# Create TOTAL buckets
games["total_bucket"] = pd.cut(
    games["TOTAL"],
    bins=[-np.inf, 144.5, 157.5, np.inf],
    labels=["144.5 and below", "145 to 157.5", "158 and above"],
    right=True
)

# Totals
total_games = games.groupby("total_bucket").size()
fav_wins = games[games["FAV_WIN"] == 1].groupby("total_bucket").size()
joint = games[games["FAV_WIN_OPP_OVER"] == 1].groupby("total_bucket").size()

# Conditional probability
prob = joint / fav_wins

# Fair American conversion
fair_price = prob.apply(lambda p: -100*p/(1-p) if p > 0.5 else 100*(1-p)/p)

# Example anticorrelation boost (10%)
boost = 0.10
boosted_prob = prob * (1 - boost)
boosted_price = boosted_prob.apply(lambda p: -100*p/(1-p) if p > 0.5 else 100*(1-p)/p)

result = pd.DataFrame({
    "Total Games": total_games,
    "Total Fav Wins": fav_wins,
    "Total Fav Wins + Opp Over": joint,
    "P(Fav Win + Opp Over | Fav Win)": prob.round(4),
    "Fair American Price": fair_price.round(0),
    "Fair American Price (anticorrelation boost)": boosted_price.round(0)
}).fillna(0)

# Clean integer columns
for c in ["Total Games", "Total Fav Wins", "Total Fav Wins + Opp Over"]:
    result[c] = result[c].astype(int)

print(result)

                 Total Games  Total Fav Wins  Total Fav Wins + Opp Over  \
total_bucket                                                              
144.5 and below         2913            2090                        808   
145 to 157.5            2413            1793                        735   
158 and above            434             349                        156   

                 P(Fav Win + Opp Over | Fav Win)  Fair American Price  \
total_bucket                                                            
144.5 and below                           0.3866                159.0   
145 to 157.5                              0.4099                144.0   
158 and above                             0.4470                124.0   

                 Fair American Price (anticorrelation boost)  
total_bucket                                                  
144.5 and below                                        187.0  
145 to 157.5                                           171.0  
158 an

In [7]:
import pandas as pd
import numpy as np

# Load file
df = pd.read_csv("cbb_data.csv")

# Clean headers
df.columns = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()

# Ensure numeric
df["CLOSING SPREAD"] = pd.to_numeric(df["CLOSING SPREAD"], errors="coerce")
df["CLOSING TOTAL"]  = pd.to_numeric(df["CLOSING TOTAL"], errors="coerce")
df["Team W/L (1/0)"] = pd.to_numeric(df["Team W/L (1/0)"], errors="coerce")
df["Team ML + Opp Team Over Win/Loss (1/0)"] = pd.to_numeric(
    df["Team ML + Opp Team Over Win/Loss (1/0)"], errors="coerce"
)

# Favorite rows only (1 row per game)
fav = df[df["CLOSING SPREAD"] < 0].copy()

games = (
    fav.groupby("GAME-ID", as_index=False)
       .agg(
           TOTAL=("CLOSING TOTAL", "first"),
           FAV_WIN=("Team W/L (1/0)", "first"),
           FAV_WIN_OPP_OVER=("Team ML + Opp Team Over Win/Loss (1/0)", "first")
       )
)

# NEW TOTAL BUCKETS
games["total_bucket"] = pd.cut(
    games["TOTAL"],
    bins=[-np.inf, 148.5, 153.5, np.inf],
    labels=["148.5 and below", "149 to 153.5", "153.5 and above"],
    right=True
)

# Totals
total_games = games.groupby("total_bucket").size()
fav_wins = games[games["FAV_WIN"] == 1].groupby("total_bucket").size()
joint = games[games["FAV_WIN_OPP_OVER"] == 1].groupby("total_bucket").size()

# Conditional probability
prob = joint / fav_wins

# Fair American price
fair_price = prob.apply(lambda p: -100*p/(1-p) if p > 0.5 else 100*(1-p)/p)

# Anticorrelation boost (10% example)
boost = 0.10
boosted_prob = prob * (1 - boost)
boosted_price = boosted_prob.apply(lambda p: -100*p/(1-p) if p > 0.5 else 100*(1-p)/p)

result = pd.DataFrame({
    "Total Games": total_games,
    "Total Fav Wins": fav_wins,
    "Total Fav Wins + Opp Over": joint,
    "P(Fav Win + Opp Over | Fav Win)": prob.round(4),
    "Fair American Price": fair_price.round(0),
    "Fair American Price (anticorrelation boost)": boosted_price.round(0)
}).fillna(0)

# Clean integer columns
for c in ["Total Games", "Total Fav Wins", "Total Fav Wins + Opp Over"]:
    result[c] = result[c].astype(int)

print(result)

                 Total Games  Total Fav Wins  Total Fav Wins + Opp Over  \
total_bucket                                                              
148.5 and below         3939            2841                       1123   
149 to 153.5             926             698                        271   
153.5 and above          895             693                        305   

                 P(Fav Win + Opp Over | Fav Win)  Fair American Price  \
total_bucket                                                            
148.5 and below                           0.3953                153.0   
149 to 153.5                              0.3883                158.0   
153.5 and above                           0.4401                127.0   

                 Fair American Price (anticorrelation boost)  
total_bucket                                                  
148.5 and below                                        181.0  
149 to 153.5                                           186.0  
153.5 